In [1]:
import pandas as pd
import os

# ── File Paths ──────────────────────────────────────────────────────────────
BASE = "/Users/mayarabin/Desktop/Thesis Downloads V2/nhgis0001_csv"

files = {
    # STF1 — 100% count data (race, age, sex, households, housing)
    "stf1_cty_sub": os.path.join(BASE, "nhgis0001_ds120_1990_cty_sub.csv"),
    "stf1_place":   os.path.join(BASE, "nhgis0001_ds120_1990_place.csv"),

    # STF3 — sample data (income, poverty, education, disability, etc.)
    "stf3_cty_sub": os.path.join(BASE, "nhgis0001_ds123_1990_cty_sub.csv"),
    "stf3_place":   os.path.join(BASE, "nhgis0001_ds123_1990_place.csv"),

    # Congressional district (you may not need this — inspect first)
    "stf3_cd103":   os.path.join(BASE, "nhgis0001_ds123_1990_cd103rd.csv"),
}

codebooks = {
    "stf1_cty_sub": os.path.join(BASE, "nhgis0001_ds120_1990_cty_sub_codebook.txt"),
    "stf1_place":   os.path.join(BASE, "nhgis0001_ds120_1990_place_codebook.txt"),
    "stf3_cty_sub": os.path.join(BASE, "nhgis0001_ds123_1990_cty_sub_codebook.txt"),
    "stf3_place":   os.path.join(BASE, "nhgis0001_ds123_1990_place_codebook.txt"),
    "stf3_cd103":   os.path.join(BASE, "nhgis0001_ds123_1990_cd103rd_codebook.txt"),
}


In [2]:
import re
import os

# ── Codebook Parser ──────────────────────────────────────────────────────────
def parse_codebook(path):
    mapping = {}
    current_table_name = ""
    table_name_pattern = re.compile(r'Source code:\s+(\w+)')
    col_pattern = re.compile(r'^\s+([A-Z][A-Z0-9]{2,5}\d{3}):\s+(.+)$')
    with open(path, encoding="latin1") as f:
        for line in f:
            line = line.rstrip()
            tn = table_name_pattern.search(line)
            if tn:
                current_table_name = tn.group(1)
            m = col_pattern.match(line)
            if m:
                code, label = m.group(1), m.group(2).strip()
                clean = label.lower()
                clean = clean.replace(" >> ", "__")
                clean = re.sub(r'[^\w]', '_', clean)
                clean = re.sub(r'_+', '_', clean)
                clean = clean.strip('_')
                full_name = f"{current_table_name}__{clean}" if current_table_name else clean
                mapping[code] = full_name
    return mapping


# ── Load Codebooks ───────────────────────────────────────────────────────────
BASE = "/Users/mayarabin/Desktop/Thesis Downloads V2/nhgis0001_csv"

stf1_col_map = parse_codebook(os.path.join(BASE, "nhgis0001_ds120_1990_cty_sub_codebook.txt"))
stf3_col_map = parse_codebook(os.path.join(BASE, "nhgis0001_ds123_1990_cty_sub_codebook.txt"))

print(f"STF1 column mappings loaded: {len(stf1_col_map)}")
print(f"STF3 column mappings loaded: {len(stf3_col_map)}")


# ── Load Data ────────────────────────────────────────────────────────────────
dfs_raw = {
    "stf1_cty_sub": pd.read_csv(os.path.join(BASE, "nhgis0001_ds120_1990_cty_sub.csv"), encoding="latin1", low_memory=False),
    "stf1_place":   pd.read_csv(os.path.join(BASE, "nhgis0001_ds120_1990_place.csv"),   encoding="latin1", low_memory=False),
    "stf3_cty_sub": pd.read_csv(os.path.join(BASE, "nhgis0001_ds123_1990_cty_sub.csv"), encoding="latin1", low_memory=False),
    "stf3_place":   pd.read_csv(os.path.join(BASE, "nhgis0001_ds123_1990_place.csv"),   encoding="latin1", low_memory=False),
}

print("\nLoaded:")
for name, df in dfs_raw.items():
    print(f"  {name}: {df.shape[0]:,} rows x {df.shape[1]} cols")


# ── Protected Geographic Columns (never rename these) ────────────────────────
GEO_COLS = {
    "GISJOIN", "YEAR", "STUSAB", "STATE", "STATEA",
    "COUNTY", "COUNTYA", "CTY_SUB", "CTY_SUBA", "COUSUBCC",
    "PLACE", "PLACEA", "PLACECC", "PLACEDC",
    "REGIONA", "DIVISIONA", "AREALAND", "AREAWAT",
    "ANPSADPI", "FUNCSTAT", "INTPTLAT", "INTPTLNG", "PSADC",
    "CMSA", "MSA_CMSAA", "PMSAA", "URB_AREAA", "URBRURALA",
    "BLOCKA", "BLCK_GRPA", "TRACTA", "CD101A", "CD103A",
    "C_CITYA", "ANRCA", "AIANHHA", "RES_ONLYA", "TRUSTA",
    "AIANCC", "RES_TRSTA", "ZIPA",
}


# ── Safe Rename ──────────────────────────────────────────────────────────────
def safe_rename(df, col_map):
    """Only rename data columns — skip any that are protected geo columns."""
    safe_map = {
        old: new
        for old, new in col_map.items()
        if old in df.columns and old not in GEO_COLS
    }
    renamed = df.rename(columns=safe_map)
    dupes = renamed.columns[renamed.columns.duplicated()].tolist()
    if dupes:
        print(f"  WARNING duplicate columns: {dupes}")
    else:
        print(f"  OK - no duplicate columns")
    return renamed

print("\nRenaming columns...")
stf1_cty_sub = safe_rename(dfs_raw["stf1_cty_sub"], stf1_col_map)
stf1_place   = safe_rename(dfs_raw["stf1_place"],   stf1_col_map)
stf3_cty_sub = safe_rename(dfs_raw["stf3_cty_sub"], stf3_col_map)
stf3_place   = safe_rename(dfs_raw["stf3_place"],   stf3_col_map)


# ── Merge STF1 + STF3 ────────────────────────────────────────────────────────
def merge_stf(stf1, stf3, geo_cols):
    """Merge STF1 + STF3 on GISJOIN, dropping redundant geo cols from STF3."""
    stf3_keep = ["GISJOIN"] + [c for c in stf3.columns if c not in geo_cols]
    stf3_trimmed = stf3[stf3_keep]
    return stf1.merge(stf3_trimmed, on="GISJOIN", how="left")

print("\nMerging...")
place_df   = merge_stf(stf1_place,   stf3_place,   GEO_COLS)
cty_sub_df = merge_stf(stf1_cty_sub, stf3_cty_sub, GEO_COLS)

print(f"place_df:   {place_df.shape[0]:,} rows x {place_df.shape[1]} cols")
print(f"cty_sub_df: {cty_sub_df.shape[0]:,} rows x {cty_sub_df.shape[1]} cols")

assert place_df.shape[0]   == dfs_raw["stf1_place"].shape[0],   "Row count changed in place merge!"
assert cty_sub_df.shape[0] == dfs_raw["stf1_cty_sub"].shape[0], "Row count changed in cty_sub merge!"
print("\nRow counts stable - merges look clean.")


# ── Sanity Check ─────────────────────────────────────────────────────────────
print("\n-- cty_sub_df geo sample --")
geo_check = [c for c in ["GISJOIN","STATE","COUNTY","CTY_SUB","CTY_SUBA"] if c in cty_sub_df.columns]
print(cty_sub_df[geo_check].head(5).to_string())

print("\n-- place_df geo sample --")
geo_check_p = [c for c in ["GISJOIN","STATE","PLACE","PLACEA"] if c in place_df.columns]
print(place_df[geo_check_p].head(5).to_string())

print("\n-- Sample data columns (cty_sub_df) --")
print([c for c in cty_sub_df.columns if '__' in c][:10])

STF1 column mappings loaded: 397
STF3 column mappings loaded: 464

Loaded:
  stf1_cty_sub: 35,298 rows x 436 cols
  stf1_place: 23,435 rows x 435 cols
  stf3_cty_sub: 35,298 rows x 503 cols
  stf3_place: 23,435 rows x 502 cols

Renaming columns...
  OK - no duplicate columns
  OK - no duplicate columns
  OK - no duplicate columns
  OK - no duplicate columns

Merging...
place_df:   23,435 rows x 899 cols
cty_sub_df: 35,298 rows x 900 cols

Row counts stable - merges look clean.

-- cty_sub_df geo sample --
         GISJOIN    STATE   COUNTY                CTY_SUB  CTY_SUBA
0  G010001090171  Alabama  Autauga  Autaugaville division     90171
1  G010001090315  Alabama  Autauga   Billingsley division     90315
2  G010001092106  Alabama  Autauga       Marbury division     92106
3  G010001092628  Alabama  Autauga    Prattville division     92628
4  G010003090207  Alabama  Baldwin   Bay Minette division     90207

-- place_df geo sample --
     GISJOIN    STATE            PLACE  PLACEA
0  G010

In [3]:
# ── Column Explorer ──────────────────────────────────────────────────────────

# 1. All non-data (geo/context) columns — these include FIPS codes
print("=== GEO / CONTEXT COLUMNS ===")
geo_cols = [c for c in cty_sub_df.columns if '__' not in c]
for c in geo_cols:
    print(f"  {c}")

# 2. All data columns grouped by source table
print("\n=== DATA COLUMNS BY TABLE ===")
data_cols = [c for c in cty_sub_df.columns if '__' in c]
from collections import defaultdict
by_table = defaultdict(list)
for c in data_cols:
    table = c.split('__')[0]
    by_table[table].append(c)

for table, cols in sorted(by_table.items()):
    print(f"\n  [{table}] — {len(cols)} columns")
    for c in cols[:3]:  # show first 3 per table as preview
        print(f"    {c}")
    if len(cols) > 3:
        print(f"    ... and {len(cols)-3} more")

=== GEO / CONTEXT COLUMNS ===
  GISJOIN
  YEAR
  STUSAB
  ANRCA
  AIANHHA
  RES_ONLYA
  TRUSTA
  AIANCC
  RES_TRSTA
  BLOCKA
  BLCK_GRPA
  TRACTA
  CD101A
  C_CITYA
  CMSA
  COUNTY
  COUNTYA
  CTY_SUB
  CTY_SUBA
  COUSUBCC
  DIVISIONA
  MSA_CMSAA
  PLACEA
  PLACECC
  PLACEDC
  PMSAA
  REGIONA
  STATE
  STATEA
  URBRURALA
  URB_AREAA
  CD103A
  AREALAND
  AREAWAT
  ANPSADPI
  FUNCSTAT
  INTPTLAT
  INTPTLNG
  PSADC

=== DATA COLUMNS BY TABLE ===

  [NH21] — 5 columns
    NH21__0_50_or_less
    NH21__0_51_to_1_00
    NH21__1_01_to_1_50
    ... and 2 more

  [NH22] — 10 columns
    NH22__owner_occupied_0_50_or_less
    NH22__owner_occupied_0_51_to_1_00
    NH22__owner_occupied_1_01_to_1_50
    ... and 7 more

  [NH3] — 2 columns
    NH3__owner_occupied
    NH3__renter_occupied

  [NH9] — 10 columns
    NH9__owner_occupied_white
    NH9__owner_occupied_black
    NH9__owner_occupied_american_indian_eskimo_or_aleut
    ... and 7 more

  [NP1] — 1 columns
    NP1__total

  [NP10] — 10 columns


In [4]:
# Load the parquet file
places_panel = pd.read_parquet("/Users/mayarabin/Desktop/Thesis Files/places_panel_final.parquet")

In [5]:
# combines the two datasets the 1990 places and all of the 1990 demographic controls

# ── Step 1: Filter panel to 1990 only ────────────────────────────────────────
panel_1990 = places_panel[places_panel['year'] == 1990].copy()
print(f"1990 panel rows: {len(panel_1990):,}")

# ── Step 2: Split panel into place vs MCD by PLACE_ID length ─────────────────
# Drop any rows with non-numeric PLACE_IDs (1 corrupted row: Lake Point, Utah)
bad_ids = ~panel_1990['PLACE_ID'].str.isnumeric()
print(f"Dropping {bad_ids.sum()} row(s) with corrupted PLACE_ID:")
print(panel_1990[bad_ids][['PLACE_ID', 'NAME', 'state_name']].to_string())
panel_1990 = panel_1990[~bad_ids].copy()

panel_place = panel_1990[panel_1990['PLACE_ID'].str.len() == 7].copy()
panel_mcd   = panel_1990[panel_1990['PLACE_ID'].str.len() == 10].copy()
print(f"\nPlace-state rows (7-digit): {len(panel_place):,}")
print(f"MCD-state rows (10-digit):  {len(panel_mcd):,}")

# ── Step 3: Build join keys in panel ─────────────────────────────────────────
# 7-digit: state(2) + place(5)
panel_place['_STATEA'] = panel_place['PLACE_ID'].str[:2].astype(int)
panel_place['_PLACEA'] = panel_place['PLACE_ID'].str[2:].astype(int)

# 10-digit: state(2) + county(3) + cousub(5)
panel_mcd['_STATEA']  = panel_mcd['PLACE_ID'].str[:2].astype(int)
panel_mcd['_COUNTYA'] = panel_mcd['PLACE_ID'].str[2:5].astype(int)
panel_mcd['_CTY_SUBA']= panel_mcd['PLACE_ID'].str[5:].astype(int)

# ── Step 4: Build join keys in census data ────────────────────────────────────
# NHGIS stores these as integers already — just make sure types match
place_df['_STATEA']  = place_df['STATEA'].astype(int)
place_df['_PLACEA']  = place_df['PLACEA'].astype(int)

cty_sub_df['_STATEA']   = cty_sub_df['STATEA'].astype(int)
cty_sub_df['_COUNTYA']  = cty_sub_df['COUNTYA'].astype(int)
cty_sub_df['_CTY_SUBA'] = cty_sub_df['CTY_SUBA'].astype(int)

# ── Step 5: Join place states ─────────────────────────────────────────────────
place_census_cols = [c for c in place_df.columns if '__' in c] + ['_STATEA', '_PLACEA', 'GISJOIN']
joined_place = panel_place.merge(
    place_df[place_census_cols],
    left_on  = ['_STATEA', '_PLACEA'],
    right_on = ['_STATEA', '_PLACEA'],
    how='left'
)
print(f"\nJoined place rows: {len(joined_place):,}")
print(f"Matched (non-null NP1__total): {joined_place['NP1__total'].notna().sum():,}")

# ── Step 6: Join MCD states ───────────────────────────────────────────────────
mcd_census_cols = [c for c in cty_sub_df.columns if '__' in c] + ['_STATEA', '_COUNTYA', '_CTY_SUBA', 'GISJOIN']
joined_mcd = panel_mcd.merge(
    cty_sub_df[mcd_census_cols],
    left_on  = ['_STATEA', '_COUNTYA', '_CTY_SUBA'],
    right_on = ['_STATEA', '_COUNTYA', '_CTY_SUBA'],
    how='left'
)
print(f"Joined MCD rows:   {len(joined_mcd):,}")
print(f"Matched (non-null NP1__total): {joined_mcd['NP1__total'].notna().sum():,}")

# ── Step 7: Stack into one unified 1990 dataset ───────────────────────────────
census_1990 = pd.concat([joined_place, joined_mcd], ignore_index=True)
print(f"\nFinal combined 1990 dataset: {census_1990.shape[0]:,} rows x {census_1990.shape[1]} cols")

# ── Step 8: Clean up helper columns (PLACE_ID is untouched) ──────────────────
census_1990 = census_1990.drop(columns=['_STATEA', '_PLACEA', '_COUNTYA', '_CTY_SUBA'], errors='ignore')

# Verify PLACE_ID is intact
print("\n── PLACE_ID integrity check ──")
print(census_1990['PLACE_ID'].head(10).tolist())
print(f"Any nulls in PLACE_ID: {census_1990['PLACE_ID'].isna().sum()}")

# ── Step 9: Match rate summary ────────────────────────────────────────────────
total = len(census_1990)
matched = census_1990['NP1__total'].notna().sum()
unmatched = total - matched
print(f"\n── Match Rate Summary ──")
print(f"Total 1990 places:  {total:,}")
print(f"Matched to census:  {matched:,} ({matched/total*100:.1f}%)")
print(f"Unmatched:          {unmatched:,} ({unmatched/total*100:.1f}%)")

# ── Step 10: Inspect unmatched rows ──────────────────────────────────────────
print("\n── Sample unmatched rows ──")
unmatched_rows = census_1990[census_1990['NP1__total'].isna()]
print(unmatched_rows[['PLACE_ID', 'NAME', 'state_name']].head(10).to_string())

1990 panel rows: 30,724
Dropping 1 row(s) with corrupted PLACE_ID:
       PLACE_ID        NAME state_name
227436  490None  Lake Point       Utah

Place-state rows (7-digit): 17,241
MCD-state rows (10-digit):  13,482

Joined place rows: 17,241
Matched (non-null NP1__total): 13,738
Joined MCD rows:   13,482
Matched (non-null NP1__total): 12,721

Final combined 1990 dataset: 30,723 rows x 911 cols

── PLACE_ID integrity check ──
['5400364', '5400748', '5400772', '5401780', '5401900', '5401996', '5403292', '5403364', '5404204', '5404276']
Any nulls in PLACE_ID: 0

── Match Rate Summary ──
Total 1990 places:  30,723
Matched to census:  26,459 (86.1%)
Unmatched:          4,264 (13.9%)

── Sample unmatched rows ──
    PLACE_ID                     NAME      state_name
37   5413525          Carpendale town   West Virginia
163  5464228     Pleasant Valley city   West Virginia
171  5466988       Ranson corporation   West Virginia
223  5486620          White Hall town   West Virginia
228  5487892 

In [6]:
import re

# ── Helper: clean place names for matching ────────────────────────────────
def clean_name(name):
    return re.sub(r'\s+(city|town|village|borough|township|corporation|cdn|plantation)$',
                  '', str(name), flags=re.IGNORECASE).strip().lower()

# ── Build name lookup from place_df ──────────────────────────────────────
place_df['_STATEA'] = place_df['STATEA'].astype(int)
place_df['_clean_name'] = place_df['PLACE'].apply(clean_name)
census_data_cols = [c for c in place_df.columns if '__' in c]

# ── Find unmatched place rows in census_1990 ──────────────────────────────
unmatched_mask = (
    census_1990['NP1__total'].isna() &
    (census_1990['PLACE_ID'].str.len() == 7)
)
print(f"Unmatched place rows before name fix: {unmatched_mask.sum():,}")

# ── Apply name-based matches ──────────────────────────────────────────────
filled = 0
for idx, row in census_1990[unmatched_mask].iterrows():
    statea = int(row['PLACE_ID'][:2])
    clean  = clean_name(row['NAME'])
    match  = place_df[
        (place_df['_STATEA'] == statea) &
        (place_df['_clean_name'] == clean)
    ]
    if len(match) == 1:
        census_1990.loc[idx, census_data_cols] = match[census_data_cols].values[0]
        filled += 1
    elif len(match) > 1:
        # Multiple matches — take first but flag it
        census_1990.loc[idx, census_data_cols] = match[census_data_cols].iloc[0].values
        filled += 1

print(f"Filled via name match: {filled:,}")

# ── Final match rate ──────────────────────────────────────────────────────
total   = len(census_1990)
matched = census_1990['NP1__total'].notna().sum()
print(f"\n── Final Match Rate ──")
print(f"Total:    {total:,}")
print(f"Matched:  {matched:,} ({matched/total*100:.1f}%)")
print(f"Unmatched:{total-matched:,} ({(total-matched)/total*100:.1f}%)")

# ── Save to parquet ───────────────────────────────────────────────────────
OUT = "/Users/mayarabin/Desktop/Thesis Files/census_1990_joined.parquet"
census_1990.to_parquet(OUT, index=False)
print(f"\nSaved to: {OUT}")

Unmatched place rows before name fix: 3,503
Filled via name match: 396

── Final Match Rate ──
Total:    30,723
Matched:  26,855 (87.4%)
Unmatched:3,868 (12.6%)

Saved to: /Users/mayarabin/Desktop/Thesis Files/census_1990_joined.parquet


In [7]:
# ── Data Completeness Report for matched rows only ───────────────────────────
matched_df = census_1990[census_1990['NP1__total'].notna()].copy()
print(f"Matched rows: {len(matched_df):,}")

# Get all census data columns
data_cols = [c for c in matched_df.columns if '__' in c]
print(f"Total census data columns: {len(data_cols)}")

# Calculate % present for each column
completeness = matched_df[data_cols].notna().mean().sort_values(ascending=False) * 100

# Most complete
print("\n=== TOP 20 most complete columns ===")
print(completeness.head(20).round(1).to_string())

# Least complete
print("\n=== BOTTOM 20 least complete columns ===")
print(completeness.tail(20).round(1).to_string())

# Summary buckets
print("\n=== Completeness distribution ===")
bins = [0, 25, 50, 75, 90, 100]
labels = ['0-25%', '25-50%', '50-75%', '75-90%', '90-100%']
dist = pd.cut(completeness, bins=bins, labels=labels)
print(dist.value_counts().sort_index().to_string())

# By table
print("\n=== Completeness by source table ===")
by_table = {}
for c in data_cols:
    table = c.split('__')[0]
    if table not in by_table:
        by_table[table] = []
    by_table[table].append(completeness[c])

for table, vals in sorted(by_table.items()):
    avg = sum(vals) / len(vals)
    print(f"  {table}: {avg:.1f}% avg ({len(vals)} cols)")

Matched rows: 26,855
Total census data columns: 861

=== TOP 20 most complete columns ===
NP1__total                                                                               100.0
NP68__female_65_years_and_over_no_work_disability_no_mobility_or_self_care_limitation    100.0
NP70__male_in_labor_force_civilian_employed                                              100.0
NP70__male_in_labor_force_civilian_unemployed                                            100.0
NP70__male_in_labor_force_civilian_not_in_labor_force                                    100.0
NP70__female_in_labor_force_in_armed_forces                                              100.0
NP70__female_in_labor_force_civilian_employed                                            100.0
NP70__female_in_labor_force_civilian_unemployed                                          100.0
NP70__female_in_labor_force_civilian_not_in_labor_force                                  100.0
NP71__white_male_in_labor_force_in_armed_forces        

In [8]:
# ── Save final demographics dataset ──────────────────────────────────────────
OUT = "/Users/mayarabin/Desktop/Thesis Files/places_panel_final_demographics.parquet"
census_1990.to_parquet(OUT, index=False)
print(f"Saved: {OUT}")
print(f"Shape: {census_1990.shape[0]:,} rows x {census_1990.shape[1]} cols")

Saved: /Users/mayarabin/Desktop/Thesis Files/places_panel_final_demographics.parquet
Shape: 30,723 rows x 907 cols
